In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("delta_table", "oh_apm_stg.vendor_extracts.load_report_log_cpc_Stg_Ref") 

In [0]:
delta_table_name = dbutils.widgets.get("delta_table")

In [0]:
import json
import requests
from datetime import datetime
from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, StringType
from delta.tables import DeltaTable

# ✅ Get run_id cleanly
run_id = dbutils.jobs.taskValues.get(taskKey="Pre_validation", key="run_id", debugValue=None)
if not run_id:
    raise ValueError("❌ run_id could not be retrieved. Make sure it's passed from the first task.")

# Clean 'Some(123)' wrapper if present
if isinstance(run_id, str) and run_id.startswith("Some("):
    run_id = run_id.replace("Some(", "").replace(")", "")
run_id = int(run_id)

gz_files = dbutils.jobs.taskValues.get(taskKey="Pre_validation", key="gz_file_list")
if isinstance(gz_files, str): 
    gz_files = json.loads(gz_files)
if not gz_files:
    raise Exception("❌ No gz files received from Pre_validation task")

# ✅ API call setup
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = ctx.apiUrl().get()
token = ctx.apiToken().get()

headers = {"Authorization": f"Bearer {token}"}
url = f"{host}/api/2.1/jobs/runs/get?run_id={run_id}"
response = requests.get(url, headers=headers)
run_data = response.json()

# Print full response for debug
print(json.dumps(run_data, indent=2))

# ✅ Extract times
start_ts = run_data.get("start_time")
end_ts = run_data.get("end_time")

start_time_str = datetime.fromtimestamp(start_ts / 1000.0).strftime("%Y-%m-%d %H:%M:%S") if start_ts else None
end_time_str = datetime.fromtimestamp(end_ts / 1000.0).strftime("%Y-%m-%d %H:%M:%S") if end_ts else None

print(f"📌 Job ID (from API): {run_id}")
print(f"📌 Start Time (from API): {start_time_str}")
print(f"📌 End Time (from API): {end_time_str}")

df = spark.read.table(delta_table_name)

# Filter only those gz files
df_to_update = df.filter(
    (col("File_Name").isin(gz_files)) &
    (col("Start_Load_Date").isNull() | col("End_Load_Date").isNull())
)

# Add start/end times
df_updated = df_to_update.withColumn("Start_Load_Date", lit(start_time_str)) \
                         .withColumn("End_Load_Date", lit(end_time_str))

# Merge back into the original table based on File_Name
delta_table = DeltaTable.forName(spark, delta_table_name)
delta_table.alias("target").merge(
    df_updated.alias("updates"),
    "target.File_Name = updates.File_Name"
).whenMatchedUpdate(set={
    "Start_Load_Date": "updates.Start_Load_Date",
    "End_Load_Date": "updates.End_Load_Date"
}).execute()

dbutils.jobs.taskValues.set(key="start_date", value=start_time_str)
dbutils.jobs.taskValues.set(key="end_date", value=end_time_str)

print(f"✅ Updated {len(gz_files)} rows in {delta_table_name} with Start/End times.")
